In [ ]:
import sys
sys.path.insert(0,'C:\\code\\python_for_the_financial_economist\\')

"""
Magic commands
"""

%load_ext autoreload
%autoreload 2

"""
Load relevant packages
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import datetime
from scipy import stats
from scipy import optimize
from scipy.linalg import block_diag

from pandas_datareader.data import DataReader
from pandas_datareader.famafrench import FamaFrenchReader, get_available_datasets

from typing import Union, List, Tuple


import cvxpy as cp

"""
Own packages
"""

from codelib.portfolio_optimization.risk_metrics import drawdown, maxdrawdown, calculate_conditional_value_at_risk

from codelib.visualization.base import correlation_plot, fan_chart
from codelib.visualization.layout import DefaultStyle, default_colors
DefaultStyle();

from codelib.statistics.moments import corr_to_cov_matrix, cov_to_corr_matrix


from codelib.portfolio_optimization.mean_variance import portfolio_mean, portfolio_std, portfolio_variance

# Exercises - Week 8


## Problem 1: Simulating utility loss

In this exercise, we will simulate the utility loss of different portfolio optimization strategies.

__Question 1__

Generate data using the following code:

In [ ]:
# set random seed for reproducibility
np.random.seed(42)

# number of assets
num_assets = 20

# generate random beta coefficients between 0.5 and 1.5
betas = np.random.uniform(0.5, 1.5, num_assets)

# market parameters
rf = 0.02 / 12  # risk-free rate
market_return = 0.08 / 12  # expected market return
market_excess_return = market_return - rf
market_std = 0.15  / np.sqrt(12) # market standard deviation

# draw idiosyncratic std for each asset between 0.05 and 0.1
idiosyncratic_stds = np.random.uniform(0.05 / np.sqrt(12), 0.1 / np.sqrt(12), num_assets)

# draw alpha for each asset between -0.01 and 0.03
alphas = np.random.uniform(-0.01 / 12, 0.03 / 12, num_assets)

# calculate vector of expected returns and covariance matrix
mu = alphas + rf + betas * market_excess_return
mu_excess = mu - rf
cov_mat = np.diag(idiosyncratic_stds**2) + np.outer(betas, betas) * market_std**2
corr_mat = cov_to_corr_matrix(cov_mat)
vols = np.sqrt(np.diag(cov_mat))

Calculate the naive "1/N" portfolio and the mean-variance efficient portfolio with $\lambda = 5$ (using true parameters)

\begin{equation*}
    \boldsymbol{w}^* = \underset{\boldsymbol{w}}{\text{arg max }} \boldsymbol{w}^\top \boldsymbol{\mu} - \frac{\lambda}{2 } \boldsymbol{w}^\top \boldsymbol{\Sigma} \boldsymbol{w}
\end{equation*}

with the constraints

* $\boldsymbol{w} \geq 0$
* $\boldsymbol{w}^\top \boldsymbol{1} = 1$
* $\boldsymbol{w} \leq 0.2$ (max 20% allocation to a single asset)

Note that $\boldsymbol{\mu}$ is the expected return, not excess return.



__Question 2__

Calculate the utility of the two portfolios.


__Question 3__

Simulate the mean-variance loss for the two portfolios . Assume a sample size of 120. Use 10,000 simulations. What are your conclusions?

Note: We can use the normal distribution and the Wishart distribution to simulate respectively sample means and sample covariances. We have true mean and covariance of the data.

__Question 4__

Repeat the analysis with $\alpha_i = 0$ for all assets.

__Question 5__

Play around with the different input parameters and values.

### Solution

__Question 1__

In [ ]:
# define relevant risk aversion parameter
lam = 5.0

In [ ]:
# define equally weighted portfolio
w_eq = np.repeat(1 / num_assets, num_assets)

In [ ]:
# define function find the mean-variance optimal portfolio given the constraint
def calculate_mvo(mu: np.ndarray, cov_mat: np.ndarray, lam: float = 5.0, solver='CLARABEL', max_weight: float = 0.2):

    # define number of assets
    num_assets = len(mu)

    # define portfolio weights to optimize over
    w = cp.Variable(num_assets)

    # define mean-variance objective
    objective = w @ mu - 0.5*lam * cp.quad_form(w, cov_mat)

    # define problem
    # note: the constraints are feasible since num_assets * 0.2 >= 1 (we need at least 5 assets to reach a full investment)
    problem = cp.Problem(cp.Maximize(objective), constraints=[w>=0, w<=max_weight, cp.sum(w)==1])

    # solve problem
    problem.solve(solver=solver)

    if w.value is None:
        raise ValueError(f"Optimization failed with status: {problem.status}")

    return w.value

In [ ]:
# calculate the true mvo portfolio
w_mvo = calculate_mvo(mu, cov_mat, lam=5.0)

__Question 2__


In [ ]:
def calculate_ceq(mu: np.ndarray, cov_mat: np.ndarray, w: np.ndarray, lam: float = 5.0):

    """
    Calculate the certainty equivalent of a portfolio given its expected returns, covariance matrix, weights, and risk aversion.

    Parameters:
    -----------
    mu : np.ndarray
        Expected returns of the assets.
    cov_mat : np.ndarray
        Covariance matrix of the asset returns.
    w : np.ndarray
        Portfolio weights.
    lam : float
        Risk aversion parameter.

    Returns:
    -------
    float
        Certainty equivalent of the portfolio.
    """

    mu_port = portfolio_mean(w, mu)
    var_port = portfolio_variance(w, cov_mat)
    ceq = mu_port- 0.5 * lam * var_port

    return ceq

In [ ]:
def calculate_loss(mu: np.ndarray, cov_mat: np.ndarray, w: np.ndarray, lam: float = 5.0, ceq_true: float = None):

    """
    Calculates the mean-variance loss.

    Parameters
    ----------

    mu:
        Expected return vector.
    cov_mat:
        The covariance matrix.
    w:
        Portfolio weights.
    lam:
        Risk aversion parameter.
    ceq_true:
        Certainty equivalent of the true optimal portfolio. If None, it is calculated by solving the
        optimization problem. Provide it to avoid re-solving the same problem in simulations.

    Returns
    -------
    float
        Mean-variance loss.

    """

    if ceq_true is None:
        w_true = calculate_mvo(mu, cov_mat, lam)
        ceq_true = calculate_ceq(mu, cov_mat, w_true, lam)

    ceq_est = calculate_ceq(mu, cov_mat, w, lam)

    loss = ceq_true - ceq_est

    return loss

In [ ]:
calculate_ceq(mu, cov_mat, w_eq, lam)

In [ ]:
calculate_ceq(mu, cov_mat, w_mvo, lam)


__Question 3__

In [ ]:
num_sim = 10_000
sample_size = 120

# random number generator for reproducible simulations
rng = np.random.default_rng(42)

sim_mu = stats.multivariate_normal.rvs(mean=mu, cov=cov_mat / sample_size, size=num_sim, random_state=rng)
sim_mu_excess = sim_mu - rf
# note: scale = cov_mat / sample_size with df = sample_size - 1 gives E[sim_cov_mat] = (n - 1) / n * cov_mat,
# i.e. the maximum likelihood (MLE) estimator of the covariance matrix (not the unbiased sample covariance)
sim_cov_mat = stats.wishart.rvs(df = sample_size-1, scale = cov_mat / sample_size, size=num_sim, random_state=rng)
sim_corr_mat = np.array([cov_to_corr_matrix(sim_cov_mat[i]) for i in range(num_sim)])

In [ ]:
# certainty equivalent of the true optimal portfolio (solved once and reused in the loss calculations)
ceq_mvo = calculate_ceq(mu, cov_mat, w_mvo, lam)

# simulate MVO portfolios and calculate their losses
sim_w_mvo = np.array([calculate_mvo(sim_mu[i], sim_cov_mat[i], lam) for i in range(num_sim)])
sim_loss = np.array([calculate_loss(mu, cov_mat, sim_w_mvo[i], lam, ceq_true=ceq_mvo) for i in range(num_sim)])

# calculate the loss for the equally weighted portfolio
loss_eq_w = calculate_loss(mu, cov_mat, np.repeat(1 / num_assets, num_assets), lam, ceq_true=ceq_mvo)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

sns.kdeplot(sim_loss, fill=True, ax=ax, color="navy", alpha=0.5)
ax.axvline(x=np.average(sim_loss), color="darkred", linestyle="--", label="Average loss")
ax.axvline(x=loss_eq_w, color="black", linestyle="--", label="Equally weighted portfolio loss", zorder=10)

ax.legend()
ax.set_title("Distribution of Loss")
ax.set_xlabel("Loss")
ax.set_ylabel("Density")

plt.tight_layout()

plt.savefig("loss_distribution_example.png", dpi=300)

Summary statistics of the simulated losses (both portfolios are evaluated with the true parameters):

In [ ]:
def summarize_loss(
    sim_loss: np.ndarray,
    loss_eq_w: float,
    ceq_true: float
) -> pd.DataFrame:
    """Summarize losses as a proportion of the true optimal portfolio's CEQ."""

    if np.isclose(ceq_true, 0):
        raise ValueError("ceq_true must be nonzero to calculate relative losses.")

    relative_loss = np.asarray(sim_loss) / ceq_true
    relative_loss_eq_w = loss_eq_w / ceq_true

    summary = pd.Series({
        "Mean relative loss (MVO)": np.mean(relative_loss),
        "Median relative loss (MVO)": np.median(relative_loss),
        "Std of relative loss (MVO)": np.std(relative_loss),
        "5% quantile (MVO)": np.quantile(relative_loss, 0.05),
        "95% quantile (MVO)": np.quantile(relative_loss, 0.95),
        "Relative loss (1/N)": relative_loss_eq_w,
        "P(loss MVO < loss 1/N)": np.mean(sim_loss < loss_eq_w),
    })

    return summary.to_frame("Proportion of true optimal CEQ")

print(summarize_loss(sim_loss, loss_eq_w, ceq_mvo))

__Conclusions (Question 3)__

* Estimation error makes the MVO portfolio suboptimal in every simulation: the loss is always positive and the average loss is substantial relative to the certainty equivalent of the true optimal portfolio.
* The loss distribution is right-skewed. In a minority of the samples, noisy estimates of the expected returns lead to a large loss.
* With sample size 120 and true alphas that are positive on average, the estimated MVO portfolio nevertheless has a lower loss than the naive 1/N portfolio in most simulations (see the probability in the table above). The constraints ($0 \le w_i \le 0.2$) limit the damage from estimation error.
* The result depends on the signal-to-noise ratio: the benefit of optimization requires that the true differences in expected returns (the alphas) are large relative to the estimation noise.

__Question 4__

In [ ]:
# set random seed for reproducibility
np.random.seed(42)

# number of assets
num_assets = 20

# generate random beta coefficients between 0.5 and 1.5
betas = np.random.uniform(0.5, 1.5, num_assets)

# market parameters
rf = 0.02 / 12  # risk-free rate
market_return = 0.08 / 12  # expected market return
market_excess_return = market_return - rf
market_std = 0.15  / np.sqrt(12) # market standard deviation

# draw idiosyncratic std for each asset between 0.05 and 0.1
idiosyncratic_stds = np.random.uniform(0.05 / np.sqrt(12), 0.1 / np.sqrt(12), num_assets)

# set alpha directly to zero for each asset
alphas_alpha_zero = np.zeros(num_assets)

# calculate vector of expected returns (the covariance matrix is unchanged)
mu_alpha_zero = alphas_alpha_zero + rf + betas * market_excess_return
mu_excess_alpha_zero = mu_alpha_zero - rf
cov_mat = np.diag(idiosyncratic_stds**2) + np.outer(betas, betas) * market_std**2
corr_mat = cov_to_corr_matrix(cov_mat)
vols = np.sqrt(np.diag(cov_mat))

In [ ]:
# calculate mvo portfolio with no alpha
w_mvo_alpha_zero = calculate_mvo(mu_alpha_zero, cov_mat, lam=5.0)

In [ ]:
calculate_ceq(mu_alpha_zero, cov_mat, w_eq, lam)

In [ ]:
ceq_mvo_alpha_zero = calculate_ceq(mu_alpha_zero, cov_mat, w_mvo_alpha_zero, lam)
ceq_mvo_alpha_zero

In [ ]:
num_sim = 10_000
sample_size = 120

rng_alpha_zero = np.random.default_rng(42)

sim_mu_alpha_zero = stats.multivariate_normal.rvs(mean=mu_alpha_zero, cov=cov_mat / sample_size, size=num_sim, random_state=rng_alpha_zero)
sim_mu_excess_alpha_zero = sim_mu_alpha_zero - rf
# note: scale = cov_mat / sample_size with df = sample_size - 1 gives the MLE covariance estimator,
# E[sim_cov_mat_alpha_zero] = (n - 1) / n * cov_mat
sim_cov_mat_alpha_zero = stats.wishart.rvs(df = sample_size-1, scale = cov_mat / sample_size, size=num_sim, random_state=rng_alpha_zero)
sim_corr_mat_alpha_zero = np.array([cov_to_corr_matrix(sim_cov_mat_alpha_zero[i]) for i in range(num_sim)])

In [ ]:
sim_w_mvo_alpha_zero = np.array([calculate_mvo(sim_mu_alpha_zero[i], sim_cov_mat_alpha_zero[i], lam) for i in range(num_sim)])
sim_loss_alpha_zero = np.array([calculate_loss(mu_alpha_zero, cov_mat, sim_w_mvo_alpha_zero[i], lam, ceq_true=ceq_mvo_alpha_zero) for i in range(num_sim)])

loss_eq_w_alpha_zero = calculate_loss(mu_alpha_zero, cov_mat, np.repeat(1 / num_assets, num_assets), lam, ceq_true=ceq_mvo_alpha_zero)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

sns.kdeplot(sim_loss_alpha_zero, fill=True, ax=ax, color="navy", alpha=0.5)
ax.axvline(x=np.average(sim_loss_alpha_zero), color="darkred", linestyle="--", label="Average loss")
ax.axvline(x=loss_eq_w_alpha_zero, color="black", linestyle="--", label="Equally weighted portfolio loss", zorder=10)

ax.legend()
ax.set_title("Distribution of Loss (zero alpha)")
ax.set_xlabel("Loss")
ax.set_ylabel("Density")

plt.tight_layout()

plt.savefig("loss_distribution_alpha_zero_example.png", dpi=300)

Summary statistics of the loss distribution (zero alpha):

In [ ]:
print(summarize_loss(sim_loss_alpha_zero, loss_eq_w_alpha_zero, ceq_mvo_alpha_zero))

__Conclusions (Question 4)__

- With zero alpha there is no true signal in the expected returns to exploit, so the mean-variance optimizer mostly chases estimation noise in the sample means.
- The loss is still positive and right-skewed, but the advantage over the equally weighted portfolio is smaller than in Question 3: MVO beats 1/N in a smaller share of the simulations (roughly three out of four in a quick check with 2,000 simulations).
- The remaining benefit comes from the covariance structure (betas and idiosyncratic variances) and the 20% position cap, which limits the damage from noisy mean estimates.
- The value of estimating expected returns depends on the signal-to-noise ratio: it pays off when true alphas are sizeable relative to estimation error, and much less so when they are zero.

(Rerun the notebook to refresh the numbers.)

__Question 5__

Play around with the different parameters / inputs.

## Problem 2: MVO with transaction costs

Assume that the investor initially holds an equally weighted portfolio, $\boldsymbol{w}_0$.

The investor considers the mean-variance efficient portfolio with $\lambda = 5$ (using true parameters) which accounts for transaction costs.

\begin{equation*}
    \boldsymbol{w}^* = \underset{\boldsymbol{w}}{\text{arg max }} \boldsymbol{w}^\top \boldsymbol{\mu} - \frac{\lambda}{2 } \boldsymbol{w}^\top \boldsymbol{\Sigma} \boldsymbol{w} - \tau \Vert \boldsymbol{w} - \boldsymbol{w}_0 \Vert_1
\end{equation*}

with the constraints

* $\boldsymbol{w} \geq 0$
* $\boldsymbol{w}^\top \boldsymbol{1} = 1$

Note that $\Vert \boldsymbol{w} - \boldsymbol{w}_0 \Vert_1$ is the L1 norm and $\tau$ is the constant proportional transaction costs.

__Question 1__

Implement the portfolio optimization as a python function.

__Question 2__

Assume $\tau = 0.005$ and


In [ ]:
corr_mat = np.array([[1.0, 0.85, 0.5, 0.45],
                     [0.85, 1.0, 0.5, 0.45],
                     [0.5, 0.5, 1.0, 0.9],
                     [0.45, 0.45, 0.9, 1.0]])

vols = np.array([5.0, 5.0, 22.0, 22.0]) / 100.0

mu = np.array([3.2, 3.22, 8.4, 8.2]) / 100.0

cov_mat = corr_to_cov_matrix(corr_mat, vols)

Calculate the optimal portfolio with and without transaction costs. Comment on your findings.



### Solution

__Question 1__

In [ ]:
# define function find the mean-variance optimal portfolio given the constraint
def calculate_mvo_with_tc(mu: np.ndarray, cov_mat: np.ndarray, lam: float = 5.0, tau: float = 0.0, w_init: np.ndarray = None, solver: str = 'CLARABEL', verbose: bool = False):

    # define number of assets
    num_assets = len(mu)

    # initial weights
    if w_init is None:
        w_init = np.repeat(1 / num_assets, num_assets)

    # define portfolio weights to optimize over
    w = cp.Variable(num_assets)

    # turn-over
    turnover = w - w_init

    # define mean-variance objective
    objective = w @ mu - 0.5*lam * cp.quad_form(w, cov_mat) - tau * cp.norm1(turnover)

    # define problem
    problem = cp.Problem(cp.Maximize(objective), constraints=[w>=0, cp.sum(w)==1])

    # solve problem
    problem.solve(solver=solver, verbose=verbose)

    if w.value is None:
        raise ValueError(f"Optimization failed with status: {problem.status}")

    return w.value

__Question 2__

In [ ]:
tau = 0.005
lam = 5.0
num_assets = 4
init_w = np.repeat(1 / num_assets, num_assets)

w_mvo_no_tc = calculate_mvo_with_tc(mu, cov_mat, lam, 0.0, init_w, solver='CLARABEL', verbose=False)
w_mvo_w_tc = calculate_mvo_with_tc(mu, cov_mat, lam, tau, init_w, solver='CLARABEL')

In [ ]:
print("Optimal weights without TC:", np.round(w_mvo_no_tc, 4))

In [ ]:
print("Optimal weights with TC:", np.round(w_mvo_w_tc, 4))

In [ ]:
# compare turnover and net utility (after transaction costs) of the two portfolios
def calculate_net_utility(w: np.ndarray, w_init: np.ndarray, tau: float) -> float:
    return w @ mu - 0.5 * lam * w @ cov_mat @ w - tau * np.sum(np.abs(w - w_init))

results_tc = pd.DataFrame({
    "Turnover": [np.sum(np.abs(w - init_w)) for w in (w_mvo_no_tc, w_mvo_w_tc)],
    "Net utility": [calculate_net_utility(w, init_w, tau) for w in (w_mvo_no_tc, w_mvo_w_tc)]
}, index=["MVO without TC", "MVO with TC"])

print(results_tc.round(6))

In [ ]:
# optimal weights for increasing transaction costs
taus = [0.0, 0.001, 0.005, 0.01, 0.02]
w_tau = pd.DataFrame([calculate_mvo_with_tc(mu, cov_mat, lam, t, init_w) for t in taus], index=taus)
w_tau["Turnover"] = np.abs(w_tau.values - init_w).sum(axis=1)
w_tau.index.name = "tau"

print(w_tau.round(4))

__Comments__

- Without transaction costs the investor shifts a lot of wealth from the equally weighted portfolio into the two low-volatility assets (assets 1 and 2), which have the better risk-adjusted trade-off.
- With transaction costs the optimal portfolio stays closer to the initial portfolio: turnover falls, because a trade is only worthwhile if its utility gain exceeds the cost of $\tau$ per unit traded. Weights in the two high-volatility assets (assets 3 and 4) end up higher, and weights in assets 1 and 2 lower, than without costs.
- The portfolio with costs has the higher net utility (measured after transaction costs), even though its weights are close to those without costs.
- As $\tau$ increases, the optimal weights converge towards the initial portfolio and turnover goes to zero.
- Note that returns are monthly, so $\tau = 0.5\%$ per unit traded is large relative to the monthly expected return differences. This is why the effect is clearly visible.

## Problem 3: Resampling

We would like to investigate a resampling approach similar to [Michaud (1989)](https://www.jstor.org/stable/4479185) but which directly uses the mean-variance trade-off

We use the data generating process:

In [ ]:
# set random seed for reproducibility
np.random.seed(42)

# number of assets
num_assets = 20

# generate random beta coefficients between 0.5 and 1.5
betas = np.random.uniform(0.5, 1.5, num_assets)

# market parameters
rf = 0.02 / 12  # risk-free rate
market_return = 0.08 / 12  # expected market return
market_excess_return = market_return - rf
market_std = 0.15  / np.sqrt(12) # market standard deviation

# draw idiosyncratic std for each asset between 0.05 and 0.1
idiosyncratic_stds = np.random.uniform(0.05 / np.sqrt(12), 0.1 / np.sqrt(12), num_assets)

# no alpha: expected returns are fully explained by the market (CAPM)
alphas = np.zeros(num_assets)

# calculate vector of expected returns and covariance matrix
mu = alphas + rf + betas * market_excess_return
mu_excess = mu - rf
cov_mat = np.diag(idiosyncratic_stds**2) + np.outer(betas, betas) * market_std**2
corr_mat = cov_to_corr_matrix(cov_mat)
vols = np.sqrt(np.diag(cov_mat))

__Question 1__

The investor considers the mean-variance efficient portfolio with $\lambda = 5$ (using true parameters).

\begin{equation*}
    \boldsymbol{w}^* = \underset{\boldsymbol{w}}{\text{arg max }} \boldsymbol{w}^\top \boldsymbol{\mu} - \frac{\lambda}{2 } \boldsymbol{w}^\top \boldsymbol{\Sigma} \boldsymbol{w}
\end{equation*}

with the constraints

* $\boldsymbol{w} \geq 0$
* $\boldsymbol{w}^\top \boldsymbol{1} = 1$

Find the true optimal portfolio.

__Question 2__

Consider the resampling approach defined by the function


In [ ]:
# define function find the mean-variance optimal portfolio given the constraint
def calculate_mvo_resampling(mu: np.ndarray, cov_mat: np.ndarray, lam: float = 5.0, sample_size: int = 120, num_resamples: int = 100, solver: str = 'CLARABEL', rng: np.random.Generator = None):

    # random number generator used for the resampling (pass a seeded generator for reproducibility)
    if rng is None:
        rng = np.random.default_rng()

    # define number of assets
    num_assets = len(mu)

    # resampled weights
    weights_resample = []

    # define portfolio weights to optimize over
    w = cp.Variable(num_assets)

    for i in range(num_resamples):

        mu_resample = stats.multivariate_normal.rvs(mean=mu, cov=cov_mat / sample_size, random_state=rng)
        cov_mat_resample = stats.wishart.rvs(df = sample_size-1, scale = cov_mat / sample_size, random_state=rng)

        # define mean-variance objective
        objective = w @ mu_resample - 0.5 * lam * cp.quad_form(w, cov_mat_resample)

        # define problem
        problem = cp.Problem(cp.Maximize(objective), constraints=[w >= 0, cp.sum(w) == 1])

        # solve problem
        problem.solve(solver=solver)

        if w.value is None:
            raise ValueError(f"Optimization failed with status: {problem.status}")

        weights_resample.append(w.value)

    return np.average(np.array(weights_resample), axis=0)

__Faster implementation using parameters__

The function above builds a new `cp.Problem` in every resample, and `cvxpy` has to re-canonicalize it each time. This is slow when the optimization is repeated many times (here $1000 \times 100$ optimizations).

`cvxpy` can instead compile the problem once if the changing data are declared as `cp.Parameter` objects. Only the parameter values are then updated before each solve (this requires that the problem is *DPP*, disciplined parametrized programming, so that parameters and variables are not multiplied in a non-convex way).

`cp.quad_form(w, cov_mat_resample)` is not DPP when the covariance matrix is a parameter. We therefore use the Cholesky factorization $\boldsymbol{\Sigma} = \boldsymbol{L} \boldsymbol{L}^\top$, such that

\begin{equation*}
    \boldsymbol{w}^\top \boldsymbol{\Sigma} \boldsymbol{w} = \Vert \boldsymbol{L}^\top \boldsymbol{w} \Vert_2^2,
\end{equation*}

which is written as `cp.sum_squares(L.T @ w)`. The parameter $\boldsymbol{L}$ is multiplied by the variable $\boldsymbol{w}$ in an affine way, so the problem is DPP. The optimization problem is mathematically identical, so with the same random draws, we get the same weights (up to solver tolerance).

In [ ]:
def calculate_mvo_resampling_param(mu: np.ndarray, cov_mat: np.ndarray, lam: float = 5.0, sample_size: int = 120, num_resamples: int = 100, solver: str = 'CLARABEL', rng: np.random.Generator = None):

    if rng is None:
        rng = np.random.default_rng()

    num_assets = len(mu)

    # portfolio weights to optimize over and parameters that change in each resample
    w = cp.Variable(num_assets)
    mu_param = cp.Parameter(num_assets)
    chol_param = cp.Parameter((num_assets, num_assets))

    # the problem is defined (and compiled) only once
    objective = w @ mu_param - 0.5 * lam * cp.sum_squares(chol_param.T @ w)
    problem = cp.Problem(cp.Maximize(objective), constraints=[w >= 0, cp.sum(w) == 1])

    weights_resample = []

    for i in range(num_resamples):

        # same draws (and order) as in calculate_mvo_resampling
        mu_param.value = stats.multivariate_normal.rvs(mean=mu, cov=cov_mat / sample_size, random_state=rng)
        cov_mat_resample = stats.wishart.rvs(df = sample_size-1, scale = cov_mat / sample_size, random_state=rng)
        chol_param.value = np.linalg.cholesky(cov_mat_resample)

        problem.solve(solver=solver)

        if w.value is None:
            raise ValueError(f"Optimization failed with status: {problem.status}")

        weights_resample.append(w.value)

    return np.average(np.array(weights_resample), axis=0)

In [ ]:
import time

# same seed, so the two implementations use identical random draws
start = time.perf_counter()
w_resampling_1 = calculate_mvo_resampling(mu, cov_mat, lam=5.0, rng=np.random.default_rng(42))
time_1 = time.perf_counter() - start

start = time.perf_counter()
w_resampling_2 = calculate_mvo_resampling_param(mu, cov_mat, lam=5.0, rng=np.random.default_rng(42))
time_2 = time.perf_counter() - start

print(f"Max abs. difference in weights: {np.max(np.abs(w_resampling_1 - w_resampling_2)):.2e}")
print(f"Time original: {time_1:.2f}s, time with parameters: {time_2:.2f}s")



__Question 3__

Simulate the mean-variance loss of the standard optimization approach and the resampling approach. Consider a sample size of 120. Use 1000 simulations. What do you observe?



### Solution

__Question 1__

In [ ]:
# calculate_mvo, calculate_ceq and calculate_loss are reused from Problem 1
# in this problem there is no maximum weight constraint (only w >= 0 and sum(w) = 1), so max_weight = 1.0
w_mvo = calculate_mvo(mu, cov_mat, lam=5.0, max_weight=1.0)

# true certainty equivalent, passed to the loss function to avoid re-solving the true optimization
ceq_mvo = calculate_ceq(mu, cov_mat, w_mvo, lam)

In [ ]:
# tiny negative values (e.g. -1e-10) are numerical noise from the solver, so clip at zero for display
print("Optimal weights:", np.round(np.clip(w_mvo, 0.0, None), 4))

__Question 2__

In [ ]:
rng_resampling = np.random.default_rng(42)
w_mvo_resampling = calculate_mvo_resampling_param(mu, cov_mat, lam=5.0, sample_size=120, num_resamples=100, solver='CLARABEL', rng=rng_resampling)

In [ ]:
print("Optimal weights with resampling:", np.round(w_mvo_resampling, 4))

__Question 3__

In [ ]:
num_sim = 1_000
sample_size = 120

rng_sim = np.random.default_rng(42)

sim_mu = stats.multivariate_normal.rvs(mean=mu, cov=cov_mat / sample_size, size=num_sim, random_state=rng_sim)
sim_mu_excess = sim_mu - rf
# scale = cov_mat / sample_size with df = sample_size - 1 gives the MLE covariance estimator
sim_cov_mat = stats.wishart.rvs(df = sample_size-1, scale = cov_mat / sample_size, size=num_sim, random_state=rng_sim)
sim_corr_mat = np.array([cov_to_corr_matrix(sim_cov_mat[i]) for i in range(num_sim)])

In [ ]:
# Simulate MVO optimale portfolios and losses using sample estimates of mean and covariance
sim_w_mvo = np.array([calculate_mvo(sim_mu[i], sim_cov_mat[i], lam, max_weight=1.0) for i in range(num_sim)])
sim_loss_mvo = np.array([calculate_loss(mu, cov_mat, sim_w_mvo[i], lam, ceq_true=ceq_mvo) for i in range(num_sim)])

# Simulate MVO resampling portfolios and losses
# resampling using the faster implementation with parameters
rng_resampling_sim = np.random.default_rng(123)
sim_w_mvo_resampling = np.array([calculate_mvo_resampling_param(sim_mu[i], sim_cov_mat[i], lam, rng=rng_resampling_sim) for i in range(num_sim)])
sim_loss_mvo_resampling = np.array([calculate_loss(mu, cov_mat, sim_w_mvo_resampling[i], lam, ceq_true=ceq_mvo) for i in range(num_sim)])

# loss for the equally weighted portfolio
loss_eq_w = calculate_loss(mu, cov_mat, np.repeat(1 / num_assets, num_assets), lam, ceq_true=ceq_mvo)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

sns.kdeplot(sim_loss_mvo, fill=True, ax=ax, color="navy", alpha=0.5)
ax.axvline(x=np.average(sim_loss_mvo), color="navy", linestyle="--", label="Average loss MVO")

sns.kdeplot(sim_loss_mvo_resampling, fill=True, ax=ax, color="darkred", alpha=0.5)
ax.axvline(x=np.average(sim_loss_mvo_resampling), color="darkred", linestyle="--", label="Average loss MVO resampling")

ax.axvline(x=loss_eq_w, color="black", linestyle="--", label="Equally weighted portfolio loss", zorder=10)

ax.legend()
ax.set_title("Distribution of Loss")
ax.set_xlabel("Loss")
ax.set_ylabel("Density")

plt.tight_layout()


Summary statistics of the loss distributions (as a proportion of the true optimal CEQ):

In [ ]:
def summarize_losses(losses: dict, loss_eq_w: float, ceq_true: float) -> pd.DataFrame:
    """Summarize the losses of several approaches as a proportion of the true optimal portfolio's CEQ."""

    summary = {}
    for name, sim_loss in losses.items():
        relative_loss = np.asarray(sim_loss) / ceq_true
        summary[name] = {
            "Mean relative loss": np.mean(relative_loss),
            "Median relative loss": np.median(relative_loss),
            "Std of relative loss": np.std(relative_loss),
            "5% quantile": np.quantile(relative_loss, 0.05),
            "95% quantile": np.quantile(relative_loss, 0.95),
            "Relative loss (1/N)": loss_eq_w / ceq_true,
            "P(loss < loss 1/N)": np.mean(np.asarray(sim_loss) < loss_eq_w),
        }

    return pd.DataFrame(summary)

print(summarize_losses({"MVO": sim_loss_mvo, "MVO resampling": sim_loss_mvo_resampling}, loss_eq_w, ceq_mvo))
print("P(loss resampling < loss MVO):", np.mean(sim_loss_mvo_resampling < sim_loss_mvo))

__Summary__

- With zero alpha there is no true signal in the expected returns, so the standard mean-variance optimizer mostly chases estimation noise in the sample means. This leads to concentrated portfolios and a large average loss.
- Resampling averages the optimal portfolios over many draws of the parameters, which gives more diversified portfolios and a clearly lower average loss than the standard approach. Resampling also beats the standard approach in most of the simulations.
- The standard optimizer is typically worse than the equally weighted portfolio (a strong benchmark here, since there is nothing to exploit in the expected returns), whereas the resampled portfolio is on average slightly better than the equally weighted portfolio.
- The benefit of resampling comes from shrinking the weights towards a diversified portfolio. Note that resampling adds a second layer of estimation noise (we resample around the estimated parameters), so it is a heuristic and not an optimal way to deal with estimation error.
- With true (non-zero) alphas the ranking could be different, because resampling also dampens real signals.


## Problem 4:  Portfolio characteristics

In this exercise, we will compare a value weighted versus an equally weighted strategy on the sector level.

__Question 1__

Obtain the Fama-French "12 industry portfolios" since 1999. Calculate a equally weighted return index and a value weighted return index.

Plot the time series of the index development.

__Question 2__

Plot a drawdown plot for the two indices.

__Question 3__

Calculate the arithmetic and geometric return.

__Question 4__

Calculate the ann. volatility. Also calculate the skewness and kurtosis.

__Question 5__

Which strategy would you prefer?


### Solution

__Question 1__

In [ ]:
reader = FamaFrenchReader("12_Industry_Portfolios",
                          start=datetime.datetime(1999, 1, 1))
industry_port = reader.read()

# print description
industry_port['DESCR']

In [ ]:
# get equally weighted
ind_eq_weighted = industry_port[1] / 100
ind_eq_weighted.columns = ind_eq_weighted.columns.str.strip()
#ind_eq_weighted.index = pd.to_datetime(ind_eq_weighted.index, format="%Y-%m")

# get market cap weighted 
ind_mc_weighted = industry_port[0] / 100
ind_mc_weighted.columns = ind_mc_weighted.columns.str.strip()
#ind_mc_weighted.index = pd.to_datetime(ind_mc_weighted.index, format="%Y%m")

# get number of companies in each sector
num_companies = industry_port[4]
num_companies.columns = num_companies.columns.str.strip()
#num_companies.index = pd.to_datetime(num_companies.index, format="%Y%m")

# get average firm size 
avg_firm_size = industry_port[5]
avg_firm_size.columns = avg_firm_size.columns.str.strip()
#avg_firm_size.index = pd.to_datetime(avg_firm_size.index, format="%Y%m")

# get sector market cap 
sector_mkt_cap  = num_companies * avg_firm_size
total_mkt_cap = sector_mkt_cap.sum(axis=1)

# get relative market cap
sector_mkt_cap_relative = sector_mkt_cap.divide(total_mkt_cap, axis="rows")

# calculate market return 
market_return = (sector_mkt_cap_relative * ind_mc_weighted).sum(axis="columns")

# calculate equal weighted return 
equal_weighted_return = (ind_mc_weighted / 12.0).sum(axis="columns")

# asset list
asset_list = ind_mc_weighted.columns

In [ ]:
mkt_index = (1 + market_return).cumprod()
eq_index = (1 + equal_weighted_return).cumprod()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

mkt_index.plot(color="navy", label="Value weighted", xlabel="")
eq_index.plot(color="darkred", label="Equally weighted", xlabel="")
ax.set_ylabel("Index value")
ax.legend();

__Question 2__



In [ ]:
dd_mkt = drawdown(mkt_index)[0]
dd_eq = drawdown(eq_index)[0]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

dd_mkt.plot(color="navy", label="Value weighted", xlabel="")
dd_eq.plot(color="darkred", label="Equally weighted", xlabel="")
ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1))
ax.set_ylabel("Drawdown")
ax.legend();

__Question 3 and 4__

In [ ]:
portfolio_descriptive = pd.DataFrame(columns=['Value weighted', 'Equally weighted'])

portfolio_descriptive.loc['Arithmetic Avg. return (ann.)', 'Value weighted'] = np.average(market_return) * 12
portfolio_descriptive.loc['Arithmetic Avg. return (ann.)', 'Equally weighted'] = np.average(equal_weighted_return) * 12

portfolio_descriptive.loc['Geometric return (ann.)', 'Value weighted'] = mkt_index.values[-1]**(12 / len(mkt_index)) - 1
portfolio_descriptive.loc['Geometric return (ann.)', 'Equally weighted'] = eq_index.values[-1]**(12 / len(eq_index)) - 1

portfolio_descriptive.loc['Volatility (ann.)', 'Value weighted'] = market_return.std() * np.sqrt(12)
portfolio_descriptive.loc['Volatility (ann.)', 'Equally weighted'] = equal_weighted_return.std() * np.sqrt(12)

portfolio_descriptive.loc['Kurtosis', 'Value weighted'] = market_return.kurt()
portfolio_descriptive.loc['Kurtosis', 'Equally weighted'] = equal_weighted_return.kurt()

portfolio_descriptive.loc['Skewness', 'Value weighted'] = market_return.skew()
portfolio_descriptive.loc['Skewness', 'Equally weighted'] = equal_weighted_return.skew()


In [ ]:
print(portfolio_descriptive)

__Question 5__

?

## Problem 5: Satisfaction and portfolio optimization [Optional]

For simplicity, we consider the case where the linear returns follow a multivariate normal distribution 

$$
\mathbf{R} \sim MVN(\boldsymbol{\mu}, \boldsymbol{\Sigma})
$$

Furthermore, assume that the investor's utility is given as a function of the wealth one year in the future $\Psi_1$ (exponential utility)

$$
U(\Psi_1) = -e^{-\frac{1}{\gamma} \Psi_1}, \; \gamma>0,
$$

the initial wealth is $\psi_0 = 1$, and that all assets has an initial price of one. 

__Question 1__

Derive the distribution of wealth $\Psi_1$ when assuming the allocation $\boldsymbol{\alpha}$ such that the wealth after one year is given by 

$$
\Psi_1 = \boldsymbol{\alpha}^\top (1  + \mathbf{R})
$$

__Question 2__

Since the wealth $\Psi_1$ will be normally distributed, then it is possible to show that 

$$
E[-e^{-\frac{1}{\gamma} \Psi_1}] = -e^{-\frac{1}{\gamma} E[\Psi_1] + \frac{1}{2 \gamma^2}Var[\Psi_1]}
$$

The Certainty-Equivalent (CE) is given by (should look familiar)

$$
\text{CE}(\boldsymbol{\alpha}) = U^{-1}(E[-e^{-\frac{1}{\gamma} \Psi_1}]) = - \gamma \ln (-E[-e^{-\frac{1}{\gamma} \Psi_1}]) =  E[\Psi_1] - \frac{1}{2 \gamma}Var[\Psi_1]
$$

Assume $\gamma = 1$. Plot level curves, the combinations of $E[\Psi_1]$ and $Var[\Psi_1]$ that give the same utility, when the asset universe consists of 10 assets with $\boldsymbol{\mu} = (0.01, 0.02, ...,  0.10)^\top$ and $\boldsymbol{\Sigma} = Diag((0.05^2, 0.06^2, ..., 0.14^2))$. At least plot the level curve on which the equally weighted portfolio is (note that the equally weighted portfolio is defined by $\alpha_i = \psi_0 / 10$ for $i=1, ..., 10$).

__Question 3__

Find the optimal allocation

$$
\boldsymbol{\alpha}^* = \arg \max_{\boldsymbol{\alpha}} \boldsymbol{\alpha}^\top (1 + \boldsymbol{\mu}) - \frac{1}{2 \gamma} \boldsymbol{\alpha}^\top \boldsymbol{\Sigma} \boldsymbol{\alpha}
$$

subject to $\boldsymbol{\alpha}^\top \boldsymbol{1} = \psi_0$. 

Define a function calculating the optimal allocation and calculate thereafter the optimal allocation. 

Add the level curve on which we find the optimal portfolio to the plot. 

__Question 4__

Add the mean-variance efficient frontier to the plot. What to you observe?


### Solution 

__Question 1__

$$
\Psi_1 \sim N(\boldsymbol{\alpha}^\top (1  + \boldsymbol{\mu}), \boldsymbol{\alpha}^\top \Sigma \boldsymbol{\alpha})
$$

__Question 2__

In [ ]:
# number of assets
num_assets = 10

# Define the expected returns and volatilities for the assets
mu = np.arange(0.01, 0.101, 0.01)
vols = np.arange(0.05, 0.141, 0.01)
cov_mat = np.diag(vols**2)

# risk aversion parameter
gamma = 0.1

# initial wealth of the investor
initial_wealth = 1 # _000_000

In [ ]:
def calculate_port_variance(alpha: np.ndarray, cov_mat: np.ndarray): 

    """
    Calculate the variance of a portfolio given the weights and covariance matrix.
    
    Parameters
    ----------
    alpha : np.ndarray
        Portfolio weights.
    cov_mat : np.ndarray
        Covariance matrix of asset returns.
 
    Returns
    -------
    float
        Variance of the portfolio.
    """
    
    return alpha @ cov_mat @ alpha 

def calculate_port_expected_value(alpha: np.ndarray, mu: np.ndarray): 

    """
    Calculate the expected value of a portfolio given the weights and expected returns.
    
    Parameters
    ----------
    alpha : np.ndarray
        Portfolio weights.
    mu : np.ndarray
        Expected returns of the assets.
 
    Returns
    -------
    float
        Expected value of the portfolio.
    """
    
    return alpha @ (1 + mu)

def calculate_certainty_equivalent(alpha: np.ndarray,
                                   mu: np.ndarray,
                                   cov_mat: np.ndarray,
                                   gamma: float = 5.0): 

    """
    Calculate the certainty equivalent of a portfolio given the weights, expected returns, covariance matrix, and risk aversion.
    
    Parameters
    ----------
    alpha : np.ndarray
        Portfolio weights.
    mu : np.ndarray
        Expected returns of the assets.
    cov_mat : np.ndarray
        Covariance matrix of asset returns.
    gamma : float, optional
        Risk aversion parameter (default is 5.0).
 
    Returns
    -------
    float
        Certainty equivalent of the portfolio.
    """
    
    exp_wealth = calculate_port_expected_value(alpha, mu)
    var_wealth = calculate_port_variance(alpha, cov_mat)
    
    return exp_wealth - 0.5 / gamma * var_wealth

In [ ]:
w_eq = np.ones(num_assets) / num_assets
alpha_eq = w_eq * initial_wealth
eq_expected_wealth = calculate_port_expected_value(alpha_eq, mu)
eq_var = calculate_port_variance(alpha_eq, cov_mat)
eq_ce = calculate_certainty_equivalent(alpha_eq, mu, cov_mat, gamma=gamma)

In [ ]:
eq_ce

Solving for $E[\Psi_1]$ in $\text{CE}(\boldsymbol{\alpha}) = E[\Psi_1] - \frac{1}{2 \gamma}Var[\Psi_1]$ yields

$$
E[\Psi_1] = \text{CE}(\boldsymbol{\alpha}) +  \frac{1}{2 \gamma}Var[\Psi_1]
$$

which we can use to plot level curves

In [ ]:
ce_levels = [-0.25 + eq_ce, eq_ce, eq_ce + 0.25]
std_values = np.linspace(0, 0.2, 100)

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(std_values, ce_levels[0] + 0.5 / gamma * std_values**2)
ax.plot(std_values, ce_levels[1] + 0.5 / gamma * std_values**2)
ax.plot(std_values, ce_levels[2] + 0.5 / gamma * std_values**2)

ax.set(xlabel="Standard deviation of portfolio value", ylabel="Expected portfolio value");
ax.set_title("Certainty Equivalent Level Curves")

__Question 3__

From the problem

$$
\boldsymbol{\alpha}^* = \arg \max_{\boldsymbol{\alpha}} \boldsymbol{\alpha}^\top (1 + \boldsymbol{\mu}) - \frac{1}{2 \gamma} \boldsymbol{\alpha}^\top \boldsymbol{\Sigma} \boldsymbol{\alpha}
$$

subject to $\boldsymbol{\alpha}^\top \boldsymbol{1} = \psi_0$, we can write the Lagrangian ($\lambda$ is here the Lagrange multiplier)

$$
\mathcal{L}(\boldsymbol{\alpha}, \lambda) = \boldsymbol{\alpha}^\top (1 + \boldsymbol{\mu}) - \frac{1}{2 \gamma} \boldsymbol{\alpha}^\top \boldsymbol{\Sigma} \boldsymbol{\alpha} + \lambda \left(\boldsymbol{\alpha}^\top \mathbf{1} - \psi_0 \right)
$$


The first order conditions read

$$
\begin{align}
\frac{\partial \mathcal{L}(\boldsymbol{\alpha}, \lambda)}{\partial \boldsymbol{\alpha}} &=  (1 + \boldsymbol{\mu}) - \frac{1}{\gamma} \boldsymbol{\Sigma} \boldsymbol{\alpha}  + \lambda \mathbf{1} = \mathbf{0} \\
\frac{\partial \mathcal{L}(\boldsymbol{\alpha}, \lambda)}{\partial \lambda} &= \boldsymbol{\alpha}^\top \mathbf{1} - \psi_0 = 0
\end{align}
$$

From the first condition, we obtain 

$$
\boldsymbol{\alpha}^* =  \gamma \boldsymbol{\Sigma}^{-1}(1 + \boldsymbol{\mu})  + \gamma \boldsymbol{\Sigma}^{-1} \lambda \mathbf{1}
$$

Then, we obtain 

$$
\mathbf{1}^\top \boldsymbol{\alpha}^* = \gamma  \mathbf{1}^\top \boldsymbol{\Sigma}^{-1}(1 + \boldsymbol{\mu})  + \gamma \mathbf{1}^\top \boldsymbol{\Sigma}^{-1}  \mathbf{1} \lambda = \psi_0
$$

Thus, 

$$
\lambda^* = \frac{\psi_0 -  \gamma  \mathbf{1}^\top \boldsymbol{\Sigma}^{-1}(1 + \boldsymbol{\mu})}{\gamma \mathbf{1}^\top \boldsymbol{\Sigma}^{-1}  \mathbf{1}}
$$

Inserting into the optimality condition for $\boldsymbol{\alpha}$

$$
\boldsymbol{\alpha}^* =  \gamma \boldsymbol{\Sigma}^{-1}(1 + \boldsymbol{\mu})  + \gamma \boldsymbol{\Sigma}^{-1} \lambda \mathbf{1} = \gamma \boldsymbol{\Sigma}^{-1}(1 + \boldsymbol{\mu})  + \frac{\psi_0 -  \gamma  \mathbf{1}^\top \boldsymbol{\Sigma}^{-1}(1 + \boldsymbol{\mu})}{ \mathbf{1}^\top \boldsymbol{\Sigma}^{-1}  \mathbf{1}} \boldsymbol{\Sigma}^{-1}  \mathbf{1}
$$

__Note: Alternatively, one could just find the numerical solution__

In [ ]:
def calculate_optimal_alpha(mu: np.ndarray, cov_mat: np.ndarray, initial_wealth, gamma: float = 5.0): 

    """
    Calculate the optimal portfolio weights given the expected returns, covariance matrix, initial wealth, and risk aversion.
    
    Parameters
    ----------
    mu : np.ndarray
        Expected returns of the assets.
    cov_mat : np.ndarray
        Covariance matrix of asset returns.
    initial_wealth : float
        Initial wealth of the investor.
    gamma : float, optional
        Risk aversion parameter (default is 5.0).
 
    Returns
    -------
    np.ndarray
        Optimal portfolio weights.
    """
    
    cov_mat_inv = np.linalg.inv(cov_mat)
    one_vec = np.ones_like(mu)
    
    part1 = gamma*cov_mat_inv @ (1 + mu)
    
    part2 = (initial_wealth - one_vec @ part1)  * cov_mat_inv @ one_vec  / (one_vec @ cov_mat_inv @ one_vec)
    
    return part1 + part2

In [ ]:
# Calculate optimal portfolio and related metrics
alpha_opt = calculate_optimal_alpha(mu, cov_mat, initial_wealth, gamma)
opt_expected_wealth = calculate_port_expected_value(alpha_opt, mu)
opt_var = calculate_port_variance(alpha_opt, cov_mat)
opt_ce = calculate_certainty_equivalent(alpha_opt, mu, cov_mat, gamma=gamma)

In [ ]:
print(alpha_opt)

In [ ]:
"""
Plot level curves of certainty equivalent for different portfolios
"""

ce_levels = [-0.25 + eq_ce, eq_ce, eq_ce + 0.25]
std_values = np.linspace(0, 0.5, 100)

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(std_values, ce_levels[0] + 0.5 / gamma * std_values**2, color="gray", alpha=0.25)

ax.scatter(np.sqrt(eq_var), eq_expected_wealth, color="navy", label="EQ-weighted Portfolio")
ax.plot(std_values, ce_levels[1] + 0.5 / gamma * std_values**2, color="navy")

ax.plot(std_values, ce_levels[2] + 0.5 / gamma * std_values**2, color="gray", alpha=0.25)

ax.scatter(np.sqrt(opt_var), opt_expected_wealth, color="darkred", label="Optimal Portfolio")
ax.plot(std_values, opt_ce + 0.5 / gamma * std_values**2, color="darkred", linestyle="--")

ax.set(xlabel="Standard deviation of portfolio value", ylabel="Expected portfolio value");

ax.legend();

__Question 4__

Apply the formula for mean-variance optimal portfolio from the lectures. 

In [ ]:
def calculate_mvo_portfolio(mu: np.ndarray, cov_mat: np.ndarray, target: float, initial_wealth):

    """
    Calculate the mean-variance optimal portfolio given the expected returns, covariance matrix, target return, and initial wealth.
    
    Parameters
    ----------
    mu : np.ndarray
        Expected returns of the assets.
    cov_mat : np.ndarray
        Covariance matrix of asset returns.
    target : float
        Target return for the portfolio.
    initial_wealth : float
        Initial wealth of the investor.
 
    Returns
    -------
    np.ndarray
        Optimal portfolio weights.
    """
    
    b = np.r_[np.zeros_like(mu), target, initial_wealth]
    A = np.vstack((np.c_[2*cov_mat, (1 + mu), np.ones_like(mu)], 
                  np.r_[(1 + mu), 0, 0], np.r_[np.ones_like(mu), 0, 0]))
    
    alpha_opt = np.linalg.inv(A) @ b
    
    return alpha_opt[:len(mu)]

In [ ]:
"""
Calculate mean-variance efficient portfolios for a range of target returns.
"""

# Define the range of target returns for which to calculate the efficient portfolios
targets = np.linspace(initial_wealth, initial_wealth*(1 + 0.4), 50)

# Calculate the mean-variance efficient portfolios for each target return
mean_var_eff_alpha = np.array([calculate_mvo_portfolio(mu, cov_mat, t, initial_wealth)  for  t in targets])

# apply functions to calculate standard deviation and expected value for each efficient portfolio
std_eff = np.sqrt(np.apply_along_axis(calculate_port_variance, 1, mean_var_eff_alpha, (cov_mat)))
expectation_eff = np.apply_along_axis(calculate_port_expected_value, 1, mean_var_eff_alpha, (mu))

In [ ]:
"""
Plot level curves of certainty equivalent for different portfolios and the efficient frontier
"""

ce_levels = [-0.25 + eq_ce, eq_ce, eq_ce + 0.25]
std_values = np.linspace(0, 0.5, 100)

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(std_values, ce_levels[0] + 0.5 / gamma * std_values**2, color="gray", alpha=0.25)

ax.scatter(np.sqrt(eq_var), eq_expected_wealth, color="navy", label="EQ-weighted Portfolio")
ax.plot(std_values, ce_levels[1] + 0.5 / gamma * std_values**2, color="navy")

ax.plot(std_values, ce_levels[2] + 0.5 / gamma * std_values**2, color="gray", alpha=0.25)

ax.scatter(np.sqrt(opt_var), opt_expected_wealth, color="darkred", label="Optimal Portfolio")
ax.plot(std_values, opt_ce + 0.5 / gamma * std_values**2, color="darkred", linestyle="--")

ax.plot(std_eff, expectation_eff, color="black", label="MVO frontier")

ax.set(xlabel="Standard deviation of portfolio value", ylabel="Expected portfolio value");

ax.legend();

__Conclusions (Question 4)__

* The optimal portfolio from Question 3 lies on the mean-variance efficient frontier. It is the point where the highest attainable certainty-equivalent level curve is tangent to the frontier.
* The equally weighted portfolio lies strictly inside the frontier. It is dominated: a frontier portfolio exists with the same standard deviation and a higher expected value, and another with the same expected value and a lower standard deviation. Its certainty equivalent is therefore lower than the optimal one.
* Level curves further up-left correspond to higher certainty equivalents. No portfolio on or below the frontier can reach a curve above the optimal one, so the optimal portfolio maximizes expected utility.
* The frontier is the full parabola traced by the target sweep. Only the upper branch (above the global minimum-variance portfolio) is efficient.

__Comment on leverage__

* The problem has no short-selling or leverage constraint, only the budget constraint $\mathbf{1}^\top \boldsymbol{\alpha} = \psi_0$. The optimal portfolio can therefore hold large short positions in some assets and correspondingly large long positions in others, so the gross exposure $\sum_i |\alpha_i|$ can be far above the initial wealth.
* The leverage is driven by the risk aversion $\gamma$ relative to the asset variances. A low $\gamma$ (high risk tolerance) combined with small variances makes the investor willing to lever up to harvest the expected return. The low-volatility assets are often shorted to fund positions in the assets with the best return-to-risk trade-off.
* The weights are also very sensitive to the inputs $\boldsymbol{\mu}$ and $\boldsymbol{\Sigma}$ (the inverse covariance matrix amplifies estimation errors). In practice, leverage is limited by constraints such as long-only, a gross exposure cap, a maximum position size, or margin requirements, which move the solution inside the frontier.


## Problem 6:  Long-term risk and portfolio optimization

Again, consider the Fama-French "12 industry portfolios" since 1999.

__Question 1__

We may assume that log-returns follows a multivariate normal distribution. 

$$
\log \mathbf{P}_{t+1} - \log \mathbf{P}_t \sim N(\boldsymbol{\mu}, \boldsymbol{\Sigma})
$$

Estimate the expected log-return and the covariance matrix of log-returns. 

__Question 2__

Based on the above estimates, simulate the asset prices 5-years into the future. Consider 10,000 paths and yearly time steps. 

Calculate the return of an equally weighted constant mix portfolio. Plot a fan chart of the development of the index value of the constant mix. Compare with a buy and hold portfolio. 

__Question 3__

Calculate the mean, variance and CVaR at the 5-year horizon. 

__Question 4__

Select the constant mix portfolio that minimizes the 5Y CVaR(5%) for a return target of e.g. $50\%$.  

__Question 5__

Suggest and apply a resampling approach. 

__Question 6__

Do we obtain similar results with a buy and hold strategy?

### Solution 

__Question 1__

As previously seen, we can simply use `pandas-datareader`. 

In [ ]:
reader = FamaFrenchReader("12_Industry_Portfolios", start=datetime.datetime(1999, 1, 1))
industry_port = reader.read()

# print description
industry_port['DESCR']

In [ ]:
# get equally weighted
ind_eq_weighted = industry_port[1] / 100
ind_eq_weighted.columns = ind_eq_weighted.columns.str.strip()


# get market cap weighted 
ind_mc_weighted = industry_port[0] / 100
ind_mc_weighted.columns = ind_mc_weighted.columns.str.strip()

asset_list = ind_mc_weighted.columns

In [ ]:
"""
Calculate log returns
"""

log_returns = np.log(1+ind_mc_weighted)

In [ ]:
mean_log_returns = log_returns.mean()
mean_log_returns

In [ ]:
cov_mat_log_returns = log_returns.cov()
cov_mat_log_returns

In [ ]:
corr_mat_log_returns = cov_to_corr_matrix(cov_mat_log_returns)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))
correlation_plot(corr_mat_log_returns, names=asset_list, include_values=True, ax=ax)

__Question 2__

We can simulate log-returns using a multivariate normal distribution and aggregate across time. 

In [ ]:
def simulate_asset_prices(mu: np.ndarray, cov_mat: np.ndarray,
                          horizon: float, dt: float, num_sim: int,
                          transform_input: bool=False,
                          rng: np.random.Generator=None) -> np.ndarray:
    
    """
    Simulates asset prices
    
    Parameters
    ----------
    mu: 
        Expected log-returns. 
    cov_mat: 
        Covariance matrix of log-returns. 
    horizon: 
        Simulation horizon in years. 
    dt: 
        Time step. 
    num_sim: 
        Number of simulations. 
    transform_input: 
        Boolean indicating whether to transform mu and cov_mat to time_step. 
    rng: 
        Random number generator (for reproducibility). 
    
    
    Returns
    -------
    np.ndarray
        Simulated asset prices  [num sim x num time steps x num_assets]. 
    
    
    """
    
    if transform_input:
        mu = mu * dt
        cov_mat = cov_mat * dt
        
    num_assets = len(mu)
    
    num_periods = int(horizon / dt)
    
    # allocate memory
    asset_prices = np.ones((num_sim, 1 + num_periods, num_assets))
    
    # simulate returns
    if rng is None:
        rng = np.random.default_rng()
    
    log_ret = rng.multivariate_normal(mu, cov_mat, size=(num_sim, num_periods))
    
    # transform returns to asset prices
    asset_prices[:, 1:, :] = np.exp(np.cumsum(log_ret, axis=1))
    
    return asset_prices

In [ ]:
"""
Simulate asset prices
"""

num_sim = 10000
dt = 1.0 # yearly time steps (and yearly rebalancing of the constant mix)
horizon = 5.0

# the data are monthly log-returns: annualize the mean and the covariance matrix
mu = mean_log_returns.values * 12.0
cov_mat = cov_mat_log_returns.values * 12.0

rng = np.random.default_rng(seed=1234)

time_points = np.arange(0.0, horizon + 0.01, dt)
asset_prices  = simulate_asset_prices(mu, cov_mat, horizon, dt, num_sim, rng=rng)

The value of a buy and hold strategy is simply given by (assuming the initial value of the assets and portfolios are equal to 1)

$$
V_T = \mathbf{w}^\top \mathbf{P}_T
$$

where $V_T$ denotes the portfolio value, $\mathbf{P}_T$ denotes the vector of asset prices and $\mathbf{w}$ denote the initial portfolio weights. 

In [ ]:
"""
Buy and hold strategy 
"""

port_w = np.ones(len(mu)) / len(mu)

buy_and_hold_index = asset_prices @ port_w

In [ ]:
asset_prices.shape

In [ ]:
buy_and_hold_index.shape

The value of the constant mix strategy 

$$
V_T = \prod_{t = 1}^T (1 + \mathbf{w}^\top \mathbf{R}_t)
$$

where $\mathbf{R}_t$ is the period return of the assets. 

In [ ]:
"""
Constant mix porfolio (rebalanced every time step)
"""

def calculate_period_returns(index: np.ndarray) -> np.ndarray:
    
    """
    Calculates period returns
    
    Paramters
    ---------
    index: 
        Index values [num sim x num time steps x num_assets]. 
        
    Returns
    -------
    np.ndarray
        Period returns. 
    
    """
    
    # get size of matrix
    n = index.shape[1]

    # get matrix not lagged
    new_mat = index[:, 1:n]
    # get matrix lagged
    old_mat = index[:, 0:n-1]
    # get 1 + return
    periodtr = new_mat / old_mat
    
    return periodtr - 1
    

def calculate_constant_mix_index(index: np.ndarray, weights: np.ndarray) -> np.ndarray:
    
    """
    Calculates cum. return index of a constant mix strategy. 
    
    Parameters
    ----------
    index: 
        Index values  [num sim x num time steps x num_assets]. 
    weights: 
        Portfolio weights. 
        
    
    Returns
    -------
    np.ndarray
        Cum. return index [num sim x num time steps]. 
    
    """
    
    port_index = np.ones((index.shape[0], index.shape[1]))
    
    # calculate period returns on assets
    per_ret = calculate_period_returns(index)
    
    # calculate port. period return
    port_per_ret = per_ret @ weights
    
    # calculate port. index
    port_index[:, 1:] = np.cumprod(1 + port_per_ret, axis=1)
    
    return port_index
    

constant_mix_index = calculate_constant_mix_index(asset_prices, port_w)

In [ ]:
# calculate percentiles
percentiles_buy_and_hold = np.percentile(buy_and_hold_index,
                                         [0.5, 1.0, 2.5, 5, 10, 50, 90, 95, 97.5, 99.0, 99.5],
                                         axis=0)
percentiles_constant_mix = np.percentile(constant_mix_index, 
                                         [0.5, 1.0, 2.5, 5, 10, 50, 90, 95, 97.5, 99.0, 99.5],
                                         axis=0)

"""
Plot fan chart
"""

fig, ax = plt.subplots(1, 2, figsize=(16, 6))

fan_chart(time_points,
          percentiles_buy_and_hold,
          color=default_colors['cyan'],
          color_median=default_colors['dark_blue'],
          labels=['99% CI', '98% CI', '95% CI', '90% CI', '80% CI', 'Median'],
          ax=ax[0])

ax[0].set_xlabel("Year", size=15)
ax[0].set_ylabel("Index value", size=15)
ax[0].set_title("Buy and Hold", size=15)
ax[0].set_ylim([0.5, 3.0])
ax[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=5, fontsize=12);

fan_chart(time_points,
          percentiles_constant_mix,
          color=default_colors['red'],
          color_median=default_colors['dark_blue'],
          labels=['99% CI', '98% CI', '95% CI', '90% CI', '80% CI', 'Median'],
          ax=ax[1])

ax[1].set_xlabel("Year", size=15)
ax[1].set_title("Constant Mix", size=15)
ax[1].set_ylim([0.5, 3.0])
ax[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=5, fontsize=12);

__Question 3__

In [ ]:
np.std(buy_and_hold_index[:, -1])

In [ ]:
np.std(constant_mix_index[:, -1])

In [ ]:
np.mean(buy_and_hold_index[:, -1])

In [ ]:
np.mean(constant_mix_index[:, -1])

In [ ]:
buy_and_hold_index[:, -1]

In [ ]:
value_at_risk = np.percentile(buy_and_hold_index[:, -1], 5)
cond_value_at_risk = np.mean(buy_and_hold_index[:, -1][buy_and_hold_index[:, -1] <= value_at_risk])
cond_value_at_risk

In [ ]:
value_at_risk = np.percentile(constant_mix_index[:, -1], 5)
cond_value_at_risk = np.mean(constant_mix_index[:, -1][constant_mix_index[:, -1] <= value_at_risk])
cond_value_at_risk

In [ ]:
np.mean(calculate_constant_mix_index(asset_prices, port_w)[:, -1])

In [ ]:
# summary of the 5Y portfolio value (mean, variance and CVaR(5%))
summary_q3 = pd.DataFrame({name: {"Mean": np.mean(values),
                                  "Variance": np.var(values),
                                  "CVaR(5%)": calculate_conditional_value_at_risk(values, p=0.05)}
                           for name, values in {"Buy and hold": buy_and_hold_index[:, -1],
                                                "Constant mix": constant_mix_index[:, -1]}.items()})

print(summary_q3)

__Comments (Questions 2 and 3)__

* The two equally weighted strategies have very similar distributions at the 5-year horizon. The buy and hold portfolio lets winners grow, whereas the constant mix sells winners and buys losers at each (yearly) rebalancing.
* The difference between the strategies would grow with a longer horizon and more frequent rebalancing.
* (Rerun the notebook to refresh the numbers.)

__Question 4__

We can apply the techniques from the exercises in week 5.  

We need to take into account the difference between how the cum. return index of the buy and hold and constant mix portfolio is calculated. 

_Note: See [Rockafeller and Uryasev (2000)](https://www.ise.ufl.edu/uryasev/files/2011/11/CVaR1_JOR.pdf) for a more efficient approach using convex optimization in the buy-and-hold case_

In [ ]:
def objective_function(x, index, beta):

    constant_mix_index = calculate_constant_mix_index(index, x)[:, -1]

    return -calculate_conditional_value_at_risk(constant_mix_index, p = 1.0 - beta)

# define common constraints 
sum_to_one_cons = {'type': 'eq',
                   'fun' : lambda x: np.sum(x) - 1.0}

no_short_cons = {'type': 'ineq',
                 'fun' : lambda x: x}

# no short selling is imposed through the bounds (no_short_cons is an equivalent alternative)
bounds = [(0.0, 1.0)] * 12 #+ [(None, None)]


def target_return_constraint(x, target, asset_prices):

    # inequality constraint (>= 0): the expected 5Y value must be at least equal to the target
    return np.mean(calculate_constant_mix_index(asset_prices, x)[:, -1]) - target

port_weights = list()
targets = [1.4, 1.45, 1.5, 1.55, 1.6, 1.65, 1.7]
for target in targets:
    
    print(target)
    
    # define constraint 
    target_cons = {'type': 'ineq',
                   'fun' : lambda x: target_return_constraint(x, target, asset_prices)}


    x0 = np.r_[np.ones(12) / 12.0]
    res = optimize.minimize(objective_function, x0=x0, args=(asset_prices, 0.95),
                            method='SLSQP',
                            constraints=[sum_to_one_cons,  target_cons], 
                            options={'ftol': 1e-9, 'disp': True}, bounds=bounds)

    if not res.success:
        print("Warning: optimization failed for target {}: {}".format(target, res.message))

    port_weights.append(res.x) #[:-1])
    
    
port_weights = np.array(port_weights)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))


ax.stackplot(np.array(targets) - 1.0, port_weights.T, labels=asset_list);
ax.xaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=1))
ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1))
ax.legend(loc='center', bbox_to_anchor=(0.5, -0.3),
          fancybox=True, shadow=True, ncol=6);
ax.set_xlabel("Target return (5-year)")
ax.set_ylabel("Portfolio weights")
ax.set_title("Optimal weights");

In [ ]:
# equally weighted benchmark
index_vals_eq = constant_mix_index[:, -1]
print("Equally weighted: mean = {:.4f}, CVaR(5%) = {:.4f}\n".format(np.mean(index_vals_eq),
                                                               calculate_conditional_value_at_risk(index_vals_eq, p=0.05)))

for i in range(len(targets)):
    
    index_vals = calculate_constant_mix_index(asset_prices, port_weights[i])[:, -1]
    cond_value_at_risk = calculate_conditional_value_at_risk(index_vals, p=0.05)
    print("Target = {}: mean = {:.4f}, CVaR(5%) = {:.4f}".format(targets[i], np.mean(index_vals), cond_value_at_risk))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))    

for i in range(len(targets)):
    sns.kdeplot(calculate_constant_mix_index(asset_prices, port_weights[i])[:, -1],
                label="Target = {}".format(targets[i]))
    
ax.legend();

__Question 5__

We can simply resample a number of paths from the simulated asset prices and perform the portfolio optimization again. Repeat this a large number of times. The average is the optimal allocations. 

_Comment: This is a simplified approach. The purpose is not to reduce Monte Carlo noise. Each optimization uses a relatively small sample, so the CVaR is estimated from only a few tail observations (25 with 500 paths) and under-diversified portfolios are penalized: a concentrated portfolio is more likely to have a bad tail in a given subsample. Averaging the optimal weights across subsamples (bagging) then gives more diversified and more stable allocations. The dispersion of the weights across subsamples is also a useful measure of how sensitive the solution is to the scenario set._

_To some extent, this approximates estimation uncertainty when the model cannot easily be re-estimated, e.g. when the paths come from a range of stochastic models calibrated to market data with subjective parameter inputs. It is, however, only a partial approximation. If all paths come from one calibrated model, each subsample has a mean and covariance that differ from the model's by only about $1/\sqrt{n}$ sampling noise, which is much smaller than real parameter uncertainty (the standard error of a mean return is often comparable to the mean itself). The amount of "uncertainty" is then governed by the arbitrary subsample size, not by actual confidence in the inputs, and model risk is not captured. The approach is more meaningful when the simulated paths pool several models or parameter sets, especially if the subsamples are drawn so that they differ in model or regime. A simple alternative is to perturb the drift of each subsample by a random shift scaled to a plausible standard error of the mean before optimizing. A more complete approach is to resample the historical returns (or draw $\boldsymbol{\mu}$ and $\boldsymbol{\Sigma}$ from their sampling distribution) and re-simulate the paths before each optimization._

In [ ]:
targets = [1.4, 1.45, 1.5, 1.55, 1.6, 1.65]

num_resamples = 50
size_resample = 500
rng_resampling = np.random.default_rng(seed=5678)
optimal_port_resampled_weights = np.zeros((len(targets), 12))
for i in range(num_resamples): 
    print(i)
    asset_prices_sim = asset_prices[rng_resampling.integers(0, num_sim, size=size_resample), :, :]
    
    opt_port_w_sim = []
    for target in targets:

        #print(target)

        # define constraint 
        target_cons = {'type': 'ineq',
                       'fun' : lambda x: target_return_constraint(x, target, asset_prices_sim)}


        x0 = np.r_[np.ones(12) / 12.0]
        res = optimize.minimize(objective_function, x0=x0, args=(asset_prices_sim, 0.95),
                                method='SLSQP',
                                constraints=[sum_to_one_cons,  target_cons], 
                                options={'ftol': 1e-9, 'disp': False}, bounds=bounds)

        if not res.success:
            print("Warning: optimization failed for target {}: {}".format(target, res.message))

        opt_port_w_sim.append(res.x)


    optimal_port_resampled_weights += np.array(opt_port_w_sim) / num_resamples
    
optimal_port_resampled_weights = np.array(optimal_port_resampled_weights)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))


ax.stackplot(np.array(targets) - 1.0, optimal_port_resampled_weights.T, labels=asset_list);
ax.xaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=1))
ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1))
ax.legend(loc='center', bbox_to_anchor=(0.5, -0.3),
          fancybox=True, shadow=True, ncol=6);
ax.set_xlabel("Target return (5-year)")
ax.set_ylabel("Portfolio weights")
ax.set_title("Optimal weights");

In [ ]:
for i in range(len(targets)):
    
    index_vals = calculate_constant_mix_index(asset_prices, optimal_port_resampled_weights[i])[:, -1]
    cond_value_at_risk = calculate_conditional_value_at_risk(index_vals, p=0.05)
    print("Target = {}: mean = {:.4f}, CVaR(5%) = {:.4f}".format(targets[i], np.mean(index_vals), cond_value_at_risk))

__Conclusions (Questions 4 and 5)__

* With the inequality constraint, the optimizer may exceed the expected value target. For low targets the constraint is not binding, and the solution is the same minimum-CVaR portfolio (the 5Y CVaR of the value is the same for all these targets, and higher than that of the equally weighted portfolio). It is concentrated in low-volatility industries.
* Once the target exceeds the expected value of the minimum-CVaR portfolio, the constraint binds. The weights then shift towards the industries with higher expected returns and the CVaR (the average value in the worst 5% of outcomes) decreases: a higher expected return is obtained at the price of a worse tail.
* The optimal portfolios without resampling are concentrated in a few industries. The resampled portfolios are more diversified, as discussed in Question 5, and the in-sample CVaR is only slightly worse than the optimized one.
* The optimizer may report a failure (e.g. "Positive directional derivative for linesearch") for non-binding targets in the small resamples, because the objective is flat and non-smooth. The resulting weights are still reasonable, but it is a reason to prefer a convex formulation where possible.


__Question 6__

In [ ]:
def objective_function_buy_and_hold(w, index, beta):

    buy_and_hold_index = (index @ w)[:, -1]

    return -calculate_conditional_value_at_risk(buy_and_hold_index, p = 1.0 - beta)

targets = [1.4, 1.45, 1.5, 1.55, 1.6, 1.65]

num_resamples = 50
size_resample = 500
optimal_port_resampled_weights_buy_and_hold = np.zeros((len(targets), 12))
for i in range(num_resamples): 
    print(i)
    asset_prices_sim = asset_prices[rng_resampling.integers(0, num_sim, size=size_resample), :, :]
    
    opt_port_w_sim = []
    for target in targets:

        #print(target)

        # define constraint 
        target_cons = {'type': 'ineq',
                       'fun' : lambda x: np.mean((asset_prices_sim @ x)[:, -1]) - target}


        x0 = np.r_[np.ones(12) / 12.0]
        res = optimize.minimize(objective_function_buy_and_hold, x0=x0, args=(asset_prices_sim, 0.95),
                                method='SLSQP',
                                constraints=[sum_to_one_cons,  target_cons], 
                                options={'ftol': 1e-9, 'disp': False}, bounds=bounds)

        if not res.success:
            print("Warning: optimization failed for target {}: {}".format(target, res.message))

        opt_port_w_sim.append(res.x)


    optimal_port_resampled_weights_buy_and_hold += np.array(opt_port_w_sim) / num_resamples
    
optimal_port_resampled_weights_buy_and_hold = np.array(optimal_port_resampled_weights_buy_and_hold)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))


ax.stackplot(np.array(targets) - 1.0, optimal_port_resampled_weights_buy_and_hold.T, labels=asset_list);
ax.xaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=1))
ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1))
ax.legend(loc='center', bbox_to_anchor=(0.5, -0.3),
          fancybox=True, shadow=True, ncol=6);
ax.set_xlabel("Target return (5-year)")
ax.set_ylabel("Portfolio weights")
ax.set_title("Optimal weights");

__Conclusions (Question 6)__

* The buy and hold optimization is based on the weights at time 0 only, and the portfolio drifts with the asset prices. It is therefore a different problem than the constant mix, where the weights are reset at each rebalancing date.
* The optimal weights for the two strategies are qualitatively similar: low targets lead to the minimum-CVaR portfolio in low-volatility industries, and higher targets shift the weights to industries with higher expected returns. They are not identical, since the constant mix rebalances back to the weights while the buy and hold lets winners grow.
* In the buy and hold case, the CVaR of the portfolio is linear in the weights given the simulated paths, so the problem can be solved exactly as a linear program (Rockafellar and Uryasev, 2000), which is faster and more robust than SLSQP.


### Example: Implement using sampling distribution of expectation and covariance estimator of log-returns

In [ ]:
targets = [1.4, 1.45, 1.5, 1.55, 1.6, 1.65]

num_resamples = 50
num_sim_resample = 2000  # paths simulated for each re-estimated model
n_obs = len(log_returns)  # number of monthly observations used in the estimation
rng_resampling = np.random.default_rng(seed=5678)
optimal_port_param_resampled_weights = np.zeros((len(targets), 12))
for i in range(num_resamples): 
    print(i)
    # draw the sample mean and sample covariance matrix directly from their sampling distributions, assuming that
    # the observed estimates are the true parameters (normal log-returns, n monthly observations):
    # mean ~ N(mu, Sigma / n) and (n - 1) * S ~ Wishart(n - 1, Sigma), independent of each other
    mu_est = rng_resampling.multivariate_normal(mean_log_returns.values, cov_mat_log_returns.values / n_obs) * 12.0
    cov_mat_est = stats.wishart.rvs(df=n_obs - 1, scale=cov_mat_log_returns.values / (n_obs - 1),
                                    random_state=rng_resampling) * 12.0

    # simulate asset prices assuming the re-estimated parameters are the true ones
    asset_prices_sim = simulate_asset_prices(mu_est, cov_mat_est, horizon, dt, num_sim_resample, rng=rng_resampling)
    
    opt_port_w_sim = []
    for target in targets:

        #print(target)

        # define constraint 
        target_cons = {'type': 'ineq',
                       'fun' : lambda x: target_return_constraint(x, target, asset_prices_sim)}


        x0 = np.r_[np.ones(12) / 12.0]
        res = optimize.minimize(objective_function, x0=x0, args=(asset_prices_sim, 0.95),
                                method='SLSQP',
                                constraints=[sum_to_one_cons,  target_cons], 
                                options={'ftol': 1e-9, 'disp': False}, bounds=bounds)

        if not res.success:
            print("Warning: optimization failed for target {}: {}".format(target, res.message))

        opt_port_w_sim.append(res.x)


    optimal_port_param_resampled_weights += np.array(opt_port_w_sim) / num_resamples
    
optimal_port_param_resampled_weights = np.array(optimal_port_param_resampled_weights)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))


ax.stackplot(np.array(targets) - 1.0, optimal_port_param_resampled_weights.T, labels=asset_list);
ax.xaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=1))
ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1))
ax.legend(loc='center', bbox_to_anchor=(0.5, -0.3),
          fancybox=True, shadow=True, ncol=6);
ax.set_xlabel("Target return (5-year)")
ax.set_ylabel("Portfolio weights")
ax.set_title("Optimal weights");